# ResearchLanka Kaggle Main-Branch Full Run

Import this notebook into Kaggle and run cells from top to bottom.

It will:

- clone/pull the latest `main` branch
- copy your uploaded raw dataset into the repo
- install Python + Dagster dependencies
- run the Dagster no-collection preprocessing job
- audit OpenAlex and Crossref LK authorship evidence
- build best-quality embeddings
- train Logistic Regression
- train Multinomial Naive Bayes baseline
- train Linear SVM
- run formal model comparison and promote the best flat classifier
- train hierarchical field -> subfield Linear SVM
- run NMF topic modeling and evaluation
- zip outputs for download

It does **not** collect data from APIs or repositories.

## 1. Settings

In [1]:
import os
from pathlib import Path

CPU_THREADS = os.cpu_count() or 4
for variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[variable] = str(CPU_THREADS)

print("CPU threads configured:", CPU_THREADS)


REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data")
OUTPUT_ZIP = WORK_DIR / "researchlanka-kaggle-outputs.zip"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)
print("Output zip:", OUTPUT_ZIP)

CPU threads configured: 4
Repo: https://github.com/krish-anu/researchlanka-ai.git
Branch: main
Code dir: /kaggle/working/code
Backend dir: /kaggle/working/code/backend
Dataset data dir: /kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data
Output zip: /kaggle/working/researchlanka-kaggle-outputs.zip


## GPU Availability Check

Run this after enabling Kaggle Settings -> Accelerator -> GPU.

In [2]:
!nvidia-smi

Wed Sep  2 02:52:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Check Kaggle Dataset Exists

If this fails, your Kaggle dataset path is different. Update `DATASET_DATA_DIR` above.

In [3]:
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 5 -type d | head -80
!test -d {DATASET_DATA_DIR} && echo "Dataset path OK" || echo "Dataset path NOT FOUND"

total 12
drwxr-xr-x 3 root root 4096 Sep  2 02:51 .
drwxr-xr-x 8 root root 4096 Sep  2 02:51 ..
drwxr-xr-x 3 root root 4096 Sep  2 02:51 datasets
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/anusankrishnathas
/kaggle/input/datasets/anusankrishnathas/raw-data1
/kaggle/input/datasets/anusankrishnathas/raw-data1/backend
/kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data
Dataset path OK


## 3. Clone Or Pull Latest Main Branch

In [4]:
%cd /kaggle/working
if CODE_DIR.exists() and (CODE_DIR / ".git").exists():
    %cd /kaggle/working/code
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} code
    %cd /kaggle/working/code

!git log --oneline -3
!ls

/kaggle/working
Cloning into 'code'...
remote: Enumerating objects: 3756, done.
remote: Counting objects: 100% (559/559), done.
remote: Compressing objects: 100% (401/401), done.
remote: Total 3756 (delta 254), reused 246 (delta 147), pack-reused 3197 (from 2)
Receiving objects: 100% (3756/3756), 14.13 MiB | 17.28 MiB/s, done.
Resolving deltas: 100% (2237/2237), done.
/kaggle/working/code
70afd15 (HEAD -> main, origin/main, origin/feature/machine-learning, origin/HEAD) Merge pull request #497 from krish-anu/feature/machine-learning
bf950da Add ownership validation input to Makefile and update validate-ownership command
e3b773f Merge pull request #496 from krish-anu/feature/machine-learning
backend       docs		 frontend	   Makefile   README.md
CHANGELOG.md  dse-project.ipynb  KAGGLE_README.md  notebooks  scripts


## 4. Copy Uploaded Raw Data Into Backend

In [5]:
%cd /kaggle/working/code/backend
!rm -rf data
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -60

/kaggle/working/code/backend
data/config/repositories.json
data/raw/seu/oai_dc.jsonl
data/raw/sltc/oai_dc.jsonl
data/raw/jfn_medicine/html_meta.jsonl
data/raw/jfn_research/html_meta.jsonl
data/raw/cmb/rest_items.jsonl
data/raw/esn/oai_dc.jsonl
data/raw/busl/rest_items.jsonl
data/raw/sljol/crossref_works.jsonl
data/raw/sljol/crossref_collection_audit.json
data/raw/rjt/oai_dc.jsonl
data/raw/vau/oai_dc.jsonl
data/raw/nsf/rest_items.jsonl
data/raw/vpa/oai_dc.jsonl
data/raw/uom/oai_dc.jsonl
data/raw/pdn/rest_items.jsonl
data/raw/ou/oai_dc.jsonl
data/raw/sliit/oai_dc.jsonl
data/raw/ruh/oai_dc.jsonl
data/raw/openalex/openalex_sri_lanka_works.parquet
data/raw/openalex/openalex_sri_lanka_works_progress.json
data/raw/openalex/openalex_sri_lanka_pagination_audit.json
data/raw/openalex/openalex_sri_lanka_works.csv
data/raw/openalex/openalex_sri_lanka_works.jsonl
data/raw/openalex/openalex_sri_lanka_doi_conflicts.csv
data/raw/uwu/rest_items.jsonl
data/reports/dagster_collection_summary_20260804T085

## 5. Install Dependencies

Kaggle may show dependency conflict warnings. Continue if the install completes. If the session restarts, rerun from the top.

In [6]:
%cd /kaggle/working/code/backend

!python -m pip install -r requirements.txt "protobuf<6" "google-cloud-bigquery-storage>=2.30,<3"

!python -m pip install "dagster==1.13.16" "dagster-webserver==1.13.16" "protobuf<6" "google-cloud-bigquery-storage>=2.30,<3"

!python -m pip install -e . --no-deps

!python -m pip install -e dagster-quickstart --no-deps

!python -m dagster --version

import pandas as pd
import sklearn
import pyarrow as pa
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("pyarrow", pa.__version__)


/kaggle/working/code/backend
Ignoring numpy: markers 'python_version >= "3.14"' don't match your environment
Ignoring pandas: markers 'python_version >= "3.14"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of google-cloud-bigquery-storage to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 40.8 MB/s eta 0:00:00
   ━━━━

## 6. Run Dagster Pipeline Without Data Collection

This prepares existing source files and runs preprocessing through the analysis-ready dataset.

In [7]:
%cd /kaggle/working/code/backend/dagster-quickstart

import importlib
import sys
from pathlib import Path

BACKEND_DIR = Path("/kaggle/working/code/backend")
DAGSTER_SRC_DIR = BACKEND_DIR / "dagster-quickstart" / "src"
for path in (DAGSTER_SRC_DIR, BACKEND_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)
importlib.invalidate_caches()

from dagster_quickstart.definitions import defs

loaded_defs = defs() if callable(defs) else defs
job = loaded_defs.resolve_job_def("researchlanka_no_collection_preprocessing_job")
result = job.execute_in_process()
if not result.success:
    raise RuntimeError("Dagster preprocessing job failed")


/kaggle/working/code/backend/dagster-quickstart


2026-09-02 02:54:02 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - 23 - RUN_START - Started execution of run for "researchlanka_no_collection_preprocessing_job".
2026-09-02 02:54:02 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - 23 - ENGINE_EVENT - Executing steps in process (pid: 23)
2026-09-02 02:54:02 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - 23 - RESOURCE_INIT_STARTED - Starting initialization of resources [io_manager].
2026-09-02 02:54:02 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - 23 - RESOURCE_INIT_SUCCESS - Finished initialization of resources [io_manager].
2026-09-02 02:54:02 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - 23 - LOGS_CAPTURED - Ca

busl: mapped 2885 records via rest -> /kaggle/working/code/backend/data/processed/repositories/busl.jsonl
cmb: mapped 8456 records via rest -> /kaggle/working/code/backend/data/processed/repositories/cmb.jsonl
esn: mapped 0 records (raw files empty)
jfn_medicine: mapped 3766 records via html -> /kaggle/working/code/backend/data/processed/repositories/jfn_medicine.jsonl
jfn_research: mapped 11106 records via html -> /kaggle/working/code/backend/data/processed/repositories/jfn_research.jsonl
nsf: mapped 15792 records via rest -> /kaggle/working/code/backend/data/processed/repositories/nsf.jsonl
ou: mapped 3627 records via oai -> /kaggle/working/code/backend/data/processed/repositories/ou.jsonl
pdn: mapped 7740 records via rest -> /kaggle/working/code/backend/data/processed/repositories/pdn.jsonl
rjt: mapped 0 records (raw files empty)
ruh: mapped 0 records (raw files empty)
seu: mapped 6606 records via oai -> /kaggle/working/code/backend/data/processed/repositories/seu.jsonl
sliit: mappe

2026-09-02 02:54:22 +0000 - dagster - INFO - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - researchlanka_all_sources_collected - Converting 11 repository JSONL files to CSV: /kaggle/working/code/backend/data/processed/repositories_combined.csv.


uom: mapped 16540 records via oai -> /kaggle/working/code/backend/data/processed/repositories/uom.jsonl
uwu: mapped 0 records (raw files empty)
vau: mapped 0 records (raw files empty)
vpa: mapped 0 records (raw files empty)


2026-09-02 02:54:31 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - 23 - researchlanka_all_sources_collected - STEP_OUTPUT - Yielded output "result" of type "Dict[String,Any]". (Type check passed).
2026-09-02 02:54:31 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - researchlanka_all_sources_collected - Writing file at: /tmp/tmp0amkp_48/storage/researchlanka_all_sources_collected using PickledObjectFilesystemIOManager...
2026-09-02 02:54:31 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - 23 - researchlanka_all_sources_collected - ASSET_MATERIALIZATION - Materialized value researchlanka_all_sources_collected.
2026-09-02 02:54:31 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 0c92f0e6-4dd7-4880-a4f5-ab1fcf0546f9 - 23 - researchlanka_all_sources_collected - HANDLED_OUTPUT

## 7. Run LK Affiliation Audits

This checks publication-time Sri Lankan institutional authorship evidence for both OpenAlex and Crossref. The audits write review queues, verified-authorship files, issue summaries, and Markdown/PDF-ready reports under `data/reports/`.

In [8]:
%cd /kaggle/working/code/backend
!make lk-affiliation-audits PYTHON=python

from pathlib import Path
import json
import shutil
import subprocess
import pandas as pd
from IPython.display import FileLink, display

AUDITS = {
    "OpenAlex": {
        "dir": Path("data/reports/openalex_lk_affiliation_audit"),
        "summary": "lk_affiliation_audit_summary.json",
        "report": "lk_affiliation_audit_report.md",
        "verified": "verified_lk_authorships.csv",
        "review": "lk_affiliation_manual_review.csv",
        "audit_records": "lk_affiliation_audit_records.csv",
    },
    "Crossref": {
        "dir": Path("data/reports/crossref_lk_affiliation_audit"),
        "summary": "crossref_lk_affiliation_audit_summary.json",
        "report": "crossref_lk_affiliation_audit_report.md",
        "verified": "verified_lk_authorships.csv",
        "review": "crossref_lk_affiliation_manual_review.csv",
        "audit_records": "crossref_lk_affiliation_audit_records.csv",
    },
}

rows = []
for source, config in AUDITS.items():
    audit_dir = config["dir"]
    summary_path = audit_dir / config["summary"]
    report_path = audit_dir / config["report"]
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing {source} audit summary: {summary_path}")

    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    overall = summary["overall"]
    impact = summary["publication_impact"]
    problems = summary["potential_problems"]

    if source == "OpenAlex":
        total_rows = overall["currently_lk_authorships"]
        issue_key = "at_least_one_issue_authorships"
        works_key = "unique_openalex_work_ids"
        candidate = overall["currently_lk_authorships"]
    else:
        total_rows = overall["audit_rows_including_authorless_works"]
        issue_key = "at_least_one_issue_authorship_rows"
        works_key = "unique_crossref_work_ids"
        candidate = overall["candidate_lk_authorships"]

    rows.extend([
        {"source": source, "metric": "Works checked", "value": f"{overall[works_key]:,}"},
        {"source": source, "metric": "Candidate LK authorships", "value": f"{candidate:,}"},
        {"source": source, "metric": "Audited author rows", "value": f"{total_rows:,}"},
        {"source": source, "metric": "Strict verified works", "value": f"{impact['strict_verified_dataset_size']:,}"},
        {"source": source, "metric": "Retained under strict rule", "value": f"{impact['percentage_retained']}%"},
        {"source": source, "metric": "Works sent to review", "value": f"{impact['records_sent_to_review']:,}"},
        {"source": source, "metric": "Authorship rows with any issue", "value": f"{problems[issue_key]['count']:,} ({problems[issue_key]['percent']}%)"},
    ])

    print(f"{source} audit files")
    for file_name in [config["report"], config["summary"], config["audit_records"], config["review"], config["verified"]]:
        file_path = audit_dir / file_name
        size_mb = file_path.stat().st_size / (1024 * 1024) if file_path.exists() else 0
        print(f"- {file_path} ({size_mb:.2f} MB)")
    print()

    pdf_path = report_path.with_suffix(".pdf")
    if shutil.which("pandoc"):
        result = subprocess.run(
            [
                "pandoc",
                str(report_path),
                "-o",
                str(pdf_path),
                "--pdf-engine=xelatex",
                "-V",
                "geometry:margin=0.75in",
                "-V",
                "fontsize=10pt",
                "-V",
                "colorlinks=true",
            ],
            text=True,
            capture_output=True,
        )
        if result.returncode == 0:
            print(f"PDF report created: {pdf_path}")
        else:
            print(f"PDF report skipped for {source}; pandoc failed:")
            print(result.stderr[-1200:])
    else:
        print(f"PDF report skipped for {source}; pandoc is not installed in this Kaggle image.")
    print()

metrics = pd.DataFrame(rows)
display(metrics)

for source, config in AUDITS.items():
    audit_dir = config["dir"]
    display(FileLink(str(audit_dir / config["summary"])))
    display(FileLink(str(audit_dir / config["report"])))
    pdf_path = (audit_dir / config["report"]).with_suffix(".pdf")
    if pdf_path.exists():
        display(FileLink(str(pdf_path)))


/kaggle/working/code/backend
python -m src.quality.audit_openalex_lk_affiliations \
--input data/raw/openalex/openalex_sri_lanka_works.jsonl \
--output-dir data/reports/openalex_lk_affiliation_audit \
--log-level INFO \

2026-09-02 03:07:54,304 INFO openalex_lk_affiliation_audit: Building local author history from data/raw/openalex/openalex_sri_lanka_works.jsonl
2026-09-02 03:07:57,053 INFO openalex_lk_affiliation_audit: Pass 1 read 10000 works
2026-09-02 03:07:59,603 INFO openalex_lk_affiliation_audit: Pass 1 read 20000 works
2026-09-02 03:08:01,959 INFO openalex_lk_affiliation_audit: Pass 1 read 30000 works
2026-09-02 03:08:04,180 INFO openalex_lk_affiliation_audit: Pass 1 read 40000 works
2026-09-02 03:08:06,547 INFO openalex_lk_affiliation_audit: Pass 1 read 50000 works
2026-09-02 03:08:07,370 INFO openalex_lk_affiliation_audit: Author history contains 81184 authors
2026-09-02 03:08:21,062 INFO openalex_lk_affiliation_audit: Pass 2 audited 5000 works
2026-09-02 03:08:32,467 INFO op

,source,metric,value
0,OpenAlex,Works checked,"53,672"
1,OpenAlex,Candidate LK authorships,"156,473"
2,OpenAlex,Audited author rows,"156,473"
3,OpenAlex,Strict verified works,"47,102"
4,OpenAlex,Retained under strict rule,87.759%
5,OpenAlex,Works sent to review,"6,082"
6,OpenAlex,Authorship rows with any issue,"17,886 (11.4307%)"
7,Crossref,Works checked,"6,981"
8,Crossref,Candidate LK authorships,"22,665"
9,Crossref,Audited author rows,"30,046"


/kaggle/working/code/backend/data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_summary.json

/kaggle/working/code/backend/data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_report.md

/kaggle/working/code/backend/data/reports/crossref_lk_affiliation_audit/crossref_lk_affiliation_audit_summary.json

/kaggle/working/code/backend/data/reports/crossref_lk_affiliation_audit/crossref_lk_affiliation_audit_report.md

## 7. Verify Preprocessing Outputs

In [9]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv
!ls -lh data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv

import pandas as pd
paths = [
    'data/processed/common/common_publications_final.csv',
    'data/processed/common/common_publications_final_2016_2026_analysis_ready.csv',
]
for path in paths:
    frame = pd.read_csv(path, nrows=5)
    total = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(path, 'rows=', total, 'columns=', len(frame.columns))


/kaggle/working/code/backend
-rw-r--r-- 1 root root 126M Sep  2 02:54 data/processed/repositories_combined.csv
-rw-r--r-- 1 root root 49M Sep  2 02:54 data/processed/sljol.csv
-rw-r--r-- 1 root root 75M Sep  2 03:06 data/processed/common/common_publications_final.csv
-rw-r--r-- 1 root root 100M Sep  2 03:07 data/processed/common/common_publications_final_2016_2026_analysis_ready.csv
data/processed/common/common_publications_final.csv rows= 41063 columns= 67
data/processed/common/common_publications_final_2016_2026_analysis_ready.csv rows= 41063 columns= 82


## 8. Build Best-Quality Embeddings

Full text fields, trigrams, larger vocabulary, 512 dimensions, no row limit.

In [10]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python \
  EMBED_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  EMBED_MAX_FEATURES=100000 \
  EMBED_NGRAM_MAX=3 \
  EMBED_DIM=512
!ls -lh data/models/publication_text_embeddings.parquet data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings_summary.txt

/kaggle/working/code/backend
python scripts/modeling/generate_publication_text_embeddings.py --input data/processed/common/common_publications_final.csv --output data/models/publication_text_embeddings.parquet --model-output data/models/publication_text_embedding_model.joblib --manifest-output data/models/publication_text_embeddings_manifest.json --summary-output data/models/publication_text_embeddings_summary.txt --text-columns title,abstract,topics,keywords,concepts --metadata-columns record_number,publication_year,title,doi,openalex_id,source_dataset,source_institution_id,source_record_id --embedding-dim 512 --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 
Generated publication text embeddings: rows=41062, dim=512, output=data/models/publication_text_embeddings.parquet
-rw------- 1 root root 396M Sep  2 03:12 data/models/publication_text_embedding_model.joblib
-rw-r--r-- 1 root root 124M Sep  2 03:12 data/models/publication_text_embeddings.parquet
-rw------- 1 root root

## 9. Train Best-Quality Logistic Regression

In [11]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python \
  LOGREG_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  LOGREG_MAX_FEATURES=100000 \
  LOGREG_NGRAM_MAX=3 \
  LOGREG_MAX_ITER=2000
!cat data/models/logistic_regression_primary_domain_metrics.txt

/kaggle/working/code/backend
python scripts/modeling/train_logistic_regression_classifier.py --input data/processed/common/common_publications_final.csv --label-column primary_domain --text-columns title,abstract,topics,keywords,concepts --model-output data/models/logistic_regression_primary_domain.joblib --metrics-output data/models/logistic_regression_primary_domain_metrics.txt --label-counts-output data/models/logistic_regression_primary_domain_labels.csv --predictions-output data/models/logistic_regression_primary_domain_predictions.csv --manifest-output data/models/logistic_regression_primary_domain_manifest.json --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 --min-class-count 20 --test-size 0.2 --max-iter 2000 
Trained logistic_regression classifier on 40,182 rows.
Classes: 4
Accuracy: 0.8686
Balanced accuracy: 0.8602
Macro F1: 0.8578
Model: data/models/logistic_regression_primary_domain.joblib
Model SHA-256: 8311a7cee8f9549eb0ae59ec1a41fea15a8f2f76d30d633323eff957b

In [12]:
%cd /kaggle/working/code/backend
!pip install -e . --no-deps

/kaggle/working/code/backend
Obtaining file:///kaggle/working/code/backend
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for research-analytics-framework (pyproject.toml) ... done
  Created wheel for research-analytics-framework: filename=research_analytics_framework-0.1.0-0.editable-py3-none-any.whl size=8331 sha256=1e96ee4d237dd7104e2ea50f1b65e41aa620e26aeb9a5fbbfa50f2d8b9efe23e
  Stored in directory: /tmp/pip-ephem-wheel-cache-_a1j_t90/wheels/48/6f/98/3205cf08ddaa25af658fb7d96745a6be29b5c675f81653f651
Successfully built research-analytics-framework
  Attempting uninstall: research-analytics-framework
    Found existing installation: research-analytics-framework 0.1.0
    Uninstalling research-analytics-framework-0.1.0:
      Successfully uninstalled research-analytics-framework-0.1.0


## 10. Train Best-Quality Linear SVM

If Kaggle RAM fails, rerun this cell with `--max-features 50000`.

In [13]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --ngram-max 3 \
  --max-features 100000 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --max-iter 5000 \
  --test-size 0.2
!cat data/models/linear_svm_primary_domain_metrics.txt

/kaggle/working/code/backend
Trained Linear SVM classifier on 40,182 rows.
Classes: 4
Best C: 1.0
CV macro F1: 0.8691
Accuracy: 0.8853
Macro F1: 0.8738
Model: /kaggle/working/code/backend/data/models/linear_svm_primary_domain.joblib
Model SHA-256: 8f556f8956af8248c959a48eb4f13dbefb7ef5ba6b6da3de099a40997bd0ef5a
Metrics: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_metrics.txt
Predictions: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_predictions.csv
Manifest: /kaggle/working/code/backend/data/models/linear_svm_primary_domain_manifest.json
Publication Linear SVM classifier

model_family: linear_svm
input_csv: data/processed/common/common_publications_final.csv
label_column: primary_domain
text_columns: title, abstract, topics, keywords, concepts
input_rows: 41063
usable_rows: 40182
train_rows: 32145
test_rows: 8037
class_count: 4
best_C: 1.0
cv_macro_f1: 0.8691
accuracy: 0.8853
macro_f1: 0.8738
weighted_f1: 0.8852

Class distribution:
Physical 

## 11. Train Naive Bayes Baseline


In [14]:
%cd /kaggle/working/code/backend
!make train-nb PYTHON=python \
  NB_INPUT=data/processed/common/common_publications_final.csv \
  NB_LABEL_COLUMN=primary_domain \
  NB_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  NB_ALPHA=1.0 \
  NB_MIN_CLASS_COUNT=20 \
  NB_TEST_SIZE=0.2
!cat data/models/multinomial_nb_primary_domain_metrics.txt


/kaggle/working/code/backend
python -m src.modeling.training --model-family multinomial_nb --input data/processed/common/common_publications_final.csv --label-column primary_domain --text-columns title,abstract,topics,keywords,concepts --alpha 1.0 --min-class-count 20 --test-size 0.2 
Trained multinomial_nb classifier on 40,182 rows.
Classes: 4
Accuracy: 0.8136
Balanced accuracy: 0.7806
Macro F1: 0.7879
Model: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain.joblib
Model SHA-256: df5d9ddd80900dcb7d1c611d5c79e6e38811d2d309a5db644238f13ddeef59fa
Metrics: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_metrics.txt
Predictions: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_predictions.csv
Confusion matrix: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_confusion_matrix.csv
Per-class results: /kaggle/working/code/backend/data/models/multinomial_nb_primary_domain_per_class.csv
Manifest: /kaggle/work

## 12. Formal Flat Classifier Comparison

This retrains Logistic Regression and Linear SVM with the same data settings, ranks by macro F1, and copies the winner into `data/models/final/`.


In [15]:
%cd /kaggle/working/code/backend
!python scripts/modeling/compare_classification_models.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --max-features 100000 \
  --ngram-max 3 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --ranking-metric macro_f1 \
  --test-size 0.2

!cat data/models/classification_comparison/model_comparison.csv
!ls -lh data/models/final


/kaggle/working/code/backend
Compared 3 classification model families.
Ranking metric: macro_f1
Best model: linear_svm
Comparison: /kaggle/working/code/backend/data/models/classification_comparison/model_comparison.csv
Manifest: /kaggle/working/code/backend/data/models/classification_comparison/model_comparison_manifest.json
Final field model: /kaggle/working/code/backend/data/models/final/publication_field_classifier.joblib
Final manifest: /kaggle/working/code/backend/data/models/final/publication_field_classifier_manifest.json




total 8.2M
-rw------- 1 root root 8.2M Sep  2 03:20 publication_field_classifier.joblib
-rw------- 1 root root  692 Sep  2 03:20 publication_field_classifier_manifest.json
-rw------- 1 root root 1.1K Sep  2 03:20 publication_field_classifier_metrics.txt


## 13. Evaluate Prediction Files Together


In [16]:
%cd /kaggle/working/code/backend
!make evaluate-models PYTHON=python \
  EVAL_PREDICTIONS="--predictions-csv data/models/classification_comparison/multinomial_nb_primary_domain_predictions.csv --predictions-csv data/models/classification_comparison/logistic_regression_primary_domain_predictions.csv --predictions-csv data/models/classification_comparison/linear_svm_primary_domain_predictions.csv" \
  EVAL_OUTPUT_DIR=data/models/evaluation
!find data/models/evaluation -maxdepth 2 -type f -print | sort


/kaggle/working/code/backend
python -m src.modeling.evaluation --predictions-csv data/models/classification_comparison/multinomial_nb_primary_domain_predictions.csv --predictions-csv data/models/classification_comparison/logistic_regression_primary_domain_predictions.csv --predictions-csv data/models/classification_comparison/linear_svm_primary_domain_predictions.csv --output-dir data/models/evaluation 
Evaluation: multinomial_nb_primary_domain

rows: 8037
class_count: 4
accuracy: 0.8180
balanced_accuracy: 0.7815
macro_precision: 0.8104
macro_recall: 0.7815
macro_f1: 0.7909
weighted_f1: 0.8141

Per-class results:
label                        support    prec  recall      f1  most confused with
Physical Sciences               2556   0.832   0.850   0.841  Social Sciences (270)
Social Sciences                 2472   0.815   0.888   0.850  Physical Sciences (162)
Health Sciences                 1921   0.821   0.835   0.828  Social Sciences (142)
Life Sciences                   1088   0.773

## 14. Train Hierarchical Field -> Subfield Linear SVM


In [17]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_hierarchical.py \
  --input data/processed/common/common_publications_final.csv \
  --field-column primary_field \
  --subfield-column primary_subfield \
  --text-columns title,abstract,topics,keywords,concepts \
  --max-features 100000 \
  --ngram-max 3 \
  --c-value 1.0 \
  --class-weight balanced \
  --max-iter 5000 \
  --predict-output data/models/linear_svm_hierarchical_predictions.csv
from pathlib import Path

metrics_candidates = [
    Path("data/models/linear_svm_hierarchical_metrics.txt"),
    Path("data/models/linear_svm_hierarchical_field_metrics.txt"),
]
metrics_path = next((path for path in metrics_candidates if path.exists()), None)
if metrics_path is None:
    raise FileNotFoundError(
        "Missing hierarchical metrics file. Checked: "
        + ", ".join(str(path) for path in metrics_candidates)
    )
print(metrics_path.read_text())
!ls -lh data/models/linear_svm_hierarchical*


/kaggle/working/code/backend
Trained hierarchical Linear SVM on 40,182 rows.
Field classes: 26
Subfield models: 26
Field accuracy: 0.8033
Field macro F1: 0.7300
Subfield mean accuracy: 0.8833
Subfield mean macro F1: 0.7963
Field model: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_field.joblib
Subfield models: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_subfields.joblib
Metrics: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_metrics.txt
Manifest: /kaggle/working/code/backend/data/models/linear_svm_hierarchical_manifest.json
Predictions written to: data/models/linear_svm_hierarchical_predictions.csv
Publication hierarchical Linear SVM (field → subfield)

model_family: linear_svm_hierarchical
input_csv: data/processed/common/common_publications_final.csv
taxonomy_json: /kaggle/working/code/backend/category_hierarchy.json
field_column: primary_field
subfield_column: primary_subfield
text_columns: title, abstract, topics, keywords, 

## 15. Run NMF Topic Modeling And Evaluation


In [18]:
%cd /kaggle/working/code/backend

!python scripts/modeling/run_nmf_topic_modeling.py \
  --data data/processed/common/common_publications_final.csv \
  --output-dir data/processed/common/nmf \
  --k-range 8 10 12 \
  --n-words 15 \
  --naming-words 3 \
  --max-iter 1000 \
  --text-columns title abstract topics keywords concepts

!find data/processed/common/nmf -maxdepth 1 -type f -print | sort

!cat data/processed/common/nmf/nmf_k_sweep_evaluation.csv


/kaggle/working/code/backend
Loading data/processed/common/common_publications_final.csv ...
Shape: (41063, 67)

Cleaning check: 119 of 41063 rows contain Tamil/Sinhala-script characters; 1578 rows contain metadata-boilerplate phrases (abstract available / editorial / etc.).
clean=True — stripping these at the token/phrase level (rows are never dropped).

No --k given, sweeping k in [8, 10, 12] ...
k=  8  coherence_cv=0.8705  diversity=0.925  redundancy=0.015  recon_err=199.4048
k= 10  coherence_cv=0.8560  diversity=0.900  redundancy=0.014  recon_err=199.0393
k= 12  coherence_cv=0.8679  diversity=0.894  redundancy=0.012  recon_err=198.6841

Best k by coherence: 8
Cleaning report: 119 of 41063 rows had non-Latin chars, 1578 rows had boilerplate phrases. clean=True
1 rows had text before cleaning but are empty after it (e.g. a title/abstract that was entirely Tamil/boilerplate) and are excluded from the NMF fit.

k=8  coherence_cv=0.8705084596426044  diversity=0.925  redundancy=0.015
Art

In [19]:
%cd /kaggle/working/code/backend
import shutil
from pathlib import Path

aliases = {
    "data/models/linear_svm_hierarchical_subfield.joblib": "data/models/linear_svm_hierarchical_subfields.joblib",
    "data/models/linear_svm_hierarchical_field_metrics.txt": "data/models/linear_svm_hierarchical_metrics.txt",
}

for source, target in aliases.items():
    source_path = Path(source)
    target_path = Path(target)
    if source_path.exists() and not target_path.exists():
        shutil.copy2(source_path, target_path)
        print(f"Aliased {source} -> {target}")

!ls -lh data/models/linear_svm_hierarchical*

/kaggle/working/code/backend
-rw------- 1 root root  25M Sep  2 03:22 data/models/linear_svm_hierarchical_field.joblib
-rw------- 1 root root   42 Sep  2 03:22 data/models/linear_svm_hierarchical_field_predictions.csv
-rw------- 1 root root  690 Sep  2 03:22 data/models/linear_svm_hierarchical_labels.csv
-rw------- 1 root root 8.2K Sep  2 03:22 data/models/linear_svm_hierarchical_manifest.json
-rw------- 1 root root 2.7K Sep  2 03:22 data/models/linear_svm_hierarchical_metrics.txt
-rw-r--r-- 1 root root  77M Sep  2 03:23 data/models/linear_svm_hierarchical_predictions.csv
-rw------- 1 root root  99M Sep  2 03:22 data/models/linear_svm_hierarchical_subfields.joblib


## 16. Verify All Modeling Outputs


In [20]:
%cd /kaggle/working/code/backend
from pathlib import Path

required = [
    "data/models/publication_text_embeddings.parquet",
    "data/models/publication_text_embedding_model.joblib",
    "data/models/logistic_regression_primary_domain.joblib",
    "data/models/logistic_regression_primary_domain_metrics.txt",
    "data/models/multinomial_nb_primary_domain.joblib",
    "data/models/multinomial_nb_primary_domain_metrics.txt",
    "data/models/linear_svm_primary_domain.joblib",
    "data/models/linear_svm_primary_domain_metrics.txt",
    "data/models/classification_comparison/model_comparison.csv",
    "data/models/final/publication_field_classifier.joblib",
    "data/models/linear_svm_hierarchical_field.joblib",
    "data/models/linear_svm_hierarchical_subfields.joblib",
    "data/models/linear_svm_hierarchical_metrics.txt",
    "data/processed/common/nmf/nmf_k_sweep_evaluation.csv",
    "data/reports/openalex_lk_affiliation_audit/verified_lk_authorships.csv",
    "data/reports/openalex_lk_affiliation_audit/lk_affiliation_manual_review.csv",
    "data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_summary.json",
    "data/reports/openalex_lk_affiliation_audit/lk_affiliation_audit_report.md",
    "data/reports/crossref_lk_affiliation_audit/verified_lk_authorships.csv",
    "data/reports/crossref_lk_affiliation_audit/crossref_lk_affiliation_manual_review.csv",
    "data/reports/crossref_lk_affiliation_audit/crossref_lk_affiliation_audit_summary.json",
    "data/reports/crossref_lk_affiliation_audit/crossref_lk_affiliation_audit_report.md",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing modeling outputs:\n" + "\n".join(missing))

for p in required:
    path = Path(p)
    print(f"OK {p} ({path.stat().st_size / (1024*1024):.2f} MB)")

/kaggle/working/code/backend
OK data/models/publication_text_embeddings.parquet (123.59 MB)
OK data/models/publication_text_embedding_model.joblib (395.14 MB)
OK data/models/logistic_regression_primary_domain.joblib (7.56 MB)
OK data/models/logistic_regression_primary_domain_metrics.txt (0.00 MB)
OK data/models/multinomial_nb_primary_domain.joblib (5.13 MB)
OK data/models/multinomial_nb_primary_domain_metrics.txt (0.00 MB)
OK data/models/linear_svm_primary_domain.joblib (8.15 MB)
OK data/models/linear_svm_primary_domain_metrics.txt (0.00 MB)
OK data/models/classification_comparison/model_comparison.csv (0.00 MB)
OK data/models/final/publication_field_classifier.joblib (8.15 MB)
OK data/models/linear_svm_hierarchical_field.joblib (24.94 MB)
OK data/models/linear_svm_hierarchical_subfields.joblib (98.47 MB)
OK data/models/linear_svm_hierarchical_metrics.txt (0.00 MB)
OK data/processed/common/nmf/nmf_k_sweep_evaluation.csv (0.00 MB)
OK data/reports/openalex_lk_affiliation_audit/verified_l

## 17. Zip Outputs For Download


In [21]:
%cd /kaggle/working/code/backend
!rm -f /kaggle/working/researchlanka-kaggle-outputs.zip
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models data/reports
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip


/kaggle/working/code/backend
  adding: data/processed/ (stored 0%)
  adding: data/processed/repositories/ (stored 0%)
  adding: data/processed/repositories/nsf.jsonl (deflated 88%)
  adding: data/processed/repositories/sliit.jsonl (deflated 77%)
  adding: data/processed/repositories/ruh.jsonl (deflated 73%)
  adding: data/processed/repositories/uom.jsonl (deflated 77%)
  adding: data/processed/repositories/pdn.jsonl (deflated 75%)
  adding: data/processed/repositories/jfn_medicine.jsonl (deflated 93%)
  adding: data/processed/repositories/seu.jsonl (deflated 81%)
  adding: data/processed/repositories/ou.jsonl (deflated 86%)
  adding: data/processed/repositories/jfn_research.jsonl (deflated 82%)
  adding: data/processed/repositories/cmb.jsonl (deflated 77%)
  adding: data/processed/repositories/busl.jsonl (deflated 91%)
  adding: data/processed/crossref/ (stored 0%)
  adding: data/processed/crossref/crossref_sri_lanka_works.jsonl (deflated 75%)
  adding: data/processed/crossref/crossref

Download these files from the Kaggle output panel:


In [22]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/researchlanka-kaggle-outputs.zip"))


/kaggle/working/researchlanka-kaggle-outputs.zip

## 18. Optional: Create Models-Only Zip


In [23]:
import os
import zipfile
from IPython.display import FileLink, display

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"
models_zip = "/kaggle/working/researchlanka-models-only.zip"

with zipfile.ZipFile(source_zip, "r") as src:
    model_files = [
        name for name in src.namelist()
        if name.startswith("data/models/") and not name.endswith("/")
    ]
    print("Model files found:", len(model_files))
    with zipfile.ZipFile(models_zip, "w", zipfile.ZIP_DEFLATED) as dst:
        for name in model_files:
            dst.writestr(name, src.read(name))

print("Created:", models_zip)
print("Size MB:", round(os.path.getsize(models_zip) / (1024**2), 2))
display(FileLink(models_zip))


Model files found: 61
Created: /kaggle/working/researchlanka-models-only.zip
Size MB: 595.53


/kaggle/working/researchlanka-models-only.zip